# 1 — Preprocessing: splits, clips, shared folder

Run this **once**, and only on the machine that holds the DOLOS annotations. Everything
happens here: splits, download, audio, `dataset.csv`. Nobody else needs to run it.

| | |
|---|---|
| **Time** | 15 s for the splits, 2–4 h for the download |
| **Writes** | `splits/`, `splits_v2/`, `clips/`, `audio/`, `dataset.csv` — directly into the shared Drive folder |
| **Resumable** | yes: re-running skips what is already there |

Writing straight to Drive costs about a second per file, but a disconnect no longer throws
away three hours of work: what is on Drive stays, and the next run continues from there.

## 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
if not os.path.isdir("/content/DeepLearningCourse/.git"):
    !git clone -q https://github.com/mattsays/DeepLearningCourse /content/DeepLearningCourse
%cd /content/DeepLearningCourse/lie-detector
!git pull -q
import sys; sys.path.insert(0, '.')

In [ ]:
# Colab has pandas, ffmpeg and tqdm; these two it lacks.
%pip install -q yt-dlp openpyxl tqdm

In [ ]:
from pathlib import Path

# The shared folder. Add a Drive shortcut named "dolos" in your My Drive if a
# teammate owns it: this path then resolves identically for everybody.
DRIVE = Path("/content/drive/MyDrive/dolos")
for sub in ("clips", "audio", "splits"):
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)
print(DRIVE, "->", sorted(p.name for p in DRIVE.iterdir()))

## 1 — Annotations

The DOLOS spreadsheets are not in git (licence) and only this notebook needs them.
If they are already in `dataset/DOLOS/` this cell does nothing; otherwise it extracts
`dolos_annotations.zip` from the shared folder.

In [ ]:
import zipfile
from pathlib import Path

DOLOS = Path("dataset/DOLOS")
archive = DRIVE / "dolos_annotations.zip"

if (DOLOS / "Dolos.xlsx").exists():
    print("annotations already in place")
elif archive.exists():
    print("extracting", archive.name)
    with zipfile.ZipFile(archive) as handle:
        handle.extractall("dataset")
else:
    raise SystemExit(
        "No DOLOS annotations: upload dolos_annotations.zip to the shared folder first."
    )
print(sorted(p.name for p in DOLOS.iterdir()))

## 2 — Download the clips

Cuts 1,671 clips, extracts the audio, writes `dataset.csv` with the 41 MUMIN columns.
Four workers is the sweet spot; if YouTube throttles you, stop the cell, drop to `--workers 2`
and run it again — it resumes from what is already on Drive.

Two progress indicators: the `download` bar (speed, ok/err counts) and, from this cell, one
line per minute with the real number of clips sitting on Drive plus an ETA.

In [16]:
import subprocess, sys, time
sys.path.insert(0, "scripts")
from build_dataset import load_clips

if not Path("dataset/DOLOS/dolos_timestamps.csv").exists():
    where = Path.cwd()
    raise SystemExit(
        f"dataset/DOLOS/ is missing (cwd is {where}). Run the Annotations cell in "
        "section 1 first; after a reconnect re-run from cell 0, because a fresh "
        "runtime starts in /content."
    )

TOTAL = sum(1 for clip in load_clips(Path("dataset/DOLOS")) if clip.duration > 0)
initial = len(list((DRIVE / "clips").glob("*.mp4")))
print(f"{TOTAL} clips expected, {initial} already on Drive")

cmd = [sys.executable, "scripts/build_dataset.py", "--workers", "4", "--quiet",
       "--out-dir", str(DRIVE / "clips"),
       "--audio-dir", str(DRIVE / "audio"),
       "--csv", str(DRIVE / "dataset.csv")]

start, process = time.time(), subprocess.Popen(cmd)
try:
    while process.poll() is None:
        done = len(list((DRIVE / "clips").glob("*.mp4")))
        minutes = (time.time() - start) / 60
        rate = (done - initial) / minutes if minutes > 0.5 else 0.0
        eta = f"{(TOTAL - done) / rate:5.0f}" if rate > 0 else "   --"
        print(f"{done:5d}/{TOTAL}  {done / TOTAL * 100:5.1f}%  "
              f"{rate:5.1f} clip/min  ETA {eta} min", flush=True)
        time.sleep(60)
except KeyboardInterrupt:
    process.terminate()
    raise
print("exit code:", process.poll())
print("clips on Drive:", len(list((DRIVE / "clips").glob("*.mp4"))))

1671 clips expected, 1452 already on Drive
 1452/1671   86.9%    0.0 clip/min  ETA    -- min


KeyboardInterrupt: 

In [17]:
# What the run actually did, and why the rest failed.
import pandas as pd

table = pd.read_csv(DRIVE / "dataset.csv")
print(table.status.value_counts().to_string())
failed = table[table.status == "error"]
if len(failed):
    print(f"\n{len(failed)} clips failed, first five:")
    for row in failed.head(5).itertuples():
        print(f"  {row.clip_id}: {str(row.error)[:110]}")
    print("\nre-run the cell above to retry them; if they all look like the same "
          "YouTube error, lower --workers to 2")

status
ok               1452
error             219
invalid_range       7

219 clips failed, first five:
  sb_wilty_ep30_lie1: ERROR: [youtube] X9LG_kwG9uQ: It was blocked due to the claimed content by Banijay x Merzigo.
  sb_wilty_ep30_lie2: ERROR: [youtube] X9LG_kwG9uQ: It was blocked due to the claimed content by Banijay x Merzigo.
  sb_wilty_ep30_lie3: ERROR: [youtube] X9LG_kwG9uQ: It was blocked due to the claimed content by Banijay x Merzigo.
  sb_wilty_ep30_lie4: ERROR: [youtube] X9LG_kwG9uQ: It was blocked due to the claimed content by Banijay x Merzigo.
  sb_wilty_ep30_truth1: ERROR: [youtube] X9LG_kwG9uQ: It was blocked due to the claimed content by Banijay x Merzigo.

re-run the cell above to retry them; if they all look like the same YouTube error, lower --workers to 2


In [18]:
import pandas as pd

table = pd.read_csv(DRIVE / "dataset.csv")
print(table.status.value_counts().to_string())
print()
ok = int((table.status == "ok").sum())
print(f"{len(table)} annotated -> {ok} usable ({ok / len(table) * 100:.1f}%)")
print("this is the number to write in the report")

status
ok               1452
error             219
invalid_range       7

1678 annotated -> 1452 usable (86.5%)
this is the number to write in the report


## 3 — Freeze the splits

Built **after** the download, on the clips that actually exist: a protocol that references a missing clip
silently shrinks every fold, and two members would end up training on different subsets.

It reserves 15% of the subjects as test and cuts the rest into 5 folds, with a validation slice inside each.
Written straight into `splits/` on Drive, so everybody reads the same files. This is **v1**, the protocol
of the results board.


In [19]:
import subprocess, sys

subprocess.run([
    sys.executable, "scripts/make_splits.py",
    "--out-dir", str(DRIVE / "splits"),
    "--available-from", str(DRIVE / "dataset.csv"),
    "--test-fraction", "0.15",
    "--folds", "5",
    "--val-fraction", "0.15",
], check=True)
print(sorted(p.name for p in (DRIVE / "splits").iterdir()))

['manifest.json', 'official_fold1.json', 'official_fold2.json', 'official_fold3.json', 'subject_fold1.json', 'subject_fold2.json', 'subject_fold3.json', 'subject_fold4.json', 'subject_fold5.json', 'subject_test.json']


## 3b — The v2 protocol (purity-balanced, for the ablation)

v1 balances clips, deception and gender; it leaves *subject-level purity* free, so one fold can end up
made of people who always tell the truth or always lie, and become much easier than the others.
`splits_v2/` keeps the same test set (copied byte-for-byte) and re-partitions only the working subjects,
matching every fold on the share of pure-subject clips and on the mean distance from a 50% rate, while
keeping small episode components whole.

It is a **separate protocol for the ablation, not a replacement**: the results board stays on `splits/`.
Measurement and caveats: [`../docs/splits-v2.md`](../docs/splits-v2.md).


In [ ]:
# v2: same test set, byte for byte; only the working subjects are re-partitioned.
# V2_REPLICATES > 1 also writes repeated-CV draws under splits_v2/replicates/.
V2_REPLICATES = 1

import subprocess, sys

subprocess.run([
    sys.executable, "scripts/make_splits_v2.py",
    "--out-dir", str(DRIVE / "splits_v2"),
    "--test-from", str(DRIVE / "splits"),
    "--available-from", str(DRIVE / "dataset.csv"),
    "--replicates", str(V2_REPLICATES),
], check=True)
print(sorted(p.name for p in (DRIVE / "splits_v2").iterdir()))


In [ ]:
import subprocess, sys

# v1 vs v2 on the same data: purity, ICC, n_eff, episode overlap
subprocess.run([
    sys.executable, "scripts/report_splits.py",
    str(DRIVE / "splits"), str(DRIVE / "splits_v2"),
    "--data-dir", str(DRIVE),
], check=True)


## 4 — Verify, then tell the group

The last check before everybody starts: five folds, the composition of each split, and the
fingerprint of the table.

In [20]:
subprocess.run([sys.executable, "-m", "lib.dataset", "--data-dir", str(DRIVE)], check=False)

CompletedProcess(args=['/usr/bin/python3', '-m', 'lib.dataset', '--data-dir', '/content/drive/MyDrive/dolos'], returncode=0)

In [21]:
import hashlib

digest = hashlib.sha256((DRIVE / "dataset.csv").read_bytes()).hexdigest()[:12]
print(f"dataset.csv sha256:{digest}")
print("clips on drive:", len(list((DRIVE / "clips").glob("*"))))
print("audio on drive:", len(list((DRIVE / "audio").glob("*"))))

dataset.csv sha256:8fa198b1fe71
clips on drive: 1452
audio on drive: 1452


Post in the chat: **how many clips are `ok`** and the **sha256 prefix** of `dataset.csv`.
Everybody else opens `architecture_template.ipynb`, points it at the same folder, and never
touches DOLOS.

Share the folder **view-only** with the other three, with write access only on
`cached_features/`: the splits and `dataset.csv` are the protocol, and a coherent
regeneration would not be caught by the hash check.